## 6 — Results Comparison

Two-stage pipeline. Visualisation/dashboard is **out of scope for this notebook** —
it will be built as a separate Lakeview dashboard in a later step.

### Stage 1 — Ingest JMeter CSV results into `results_queries`
Autoloader streams new files from `{volume}/results/` on each run. Checkpoint
lives on the same volume, so re-running picks up only new files.

Parses combo metadata from the filename
(`{SIZE}_{SCALE}_{STRATEGY_SLUG}-{PLATFORM}-{TEST_TYPE}-report.csv`), normalises
`test_type` (`perf-test` → `power`), and derives `data_format`,
`optimization_strategy`, and `warehouse_size_type` from the strategy slug.

### Stage 2 — Derive `results_prices`
One row per distinct `(platform, warehouse_size, warehouse_size_type, region)`
seen in `results_queries`. Upserts via `MERGE` so `price_per_unit` / `units_per_hour`
values you manually set are preserved across re-runs. Defaults for new rows are
`0` — edit them to compute TCO later.

Both tables are prefixed with `results_` so `7 - Databricks - Cleanup` skips them.


In [ ]:
%run "./helpers/Helper Functions"

In [ ]:
# --- Widgets ---
dbutils.widgets.removeAll()
dbutils.widgets.text("dbx_catalog", "tpcds_sf_catalog", "Databricks Catalog")
dbutils.widgets.text("dbx_schema", "tpcds", "Databricks Schema")
dbutils.widgets.text("dbx_volume", "tpcds_raw_data", "Volume name")
dbutils.widgets.text("region", "us-east-1", "Region where tests were run")

In [ ]:
import os

params = dbutils.widgets.getAll()
catalog = params["dbx_catalog"]
schema  = params["dbx_schema"]
volume  = params["dbx_volume"]
region  = params["region"]

volume_base    = f"/Volumes/{catalog}/{schema}/{volume}"
results_path   = f"{volume_base}/results/"
checkpoint_dir = f"{volume_base}/autoloader/results/checkpoint"
schema_dir     = f"{volume_base}/autoloader/results/schema"

queries_table   = f"{catalog}.{schema}.results_queries"
prices_table = f"{catalog}.{schema}.results_prices"

os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(schema_dir, exist_ok=True)

print(f"Results source : {results_path}")
print(f"Checkpoint dir : {checkpoint_dir}")
print(f"Schema dir     : {schema_dir}")
print(f"Runs table     : {queries_table}")
print(f"Prices table   : {prices_table}")
print(f"Region         : {region}")

### Stage 1 — Autoloader ingestion into `results_queries`

In [ ]:
# --- Ensure the results_queries table exists with the canonical schema ---
# (autoloader writes into it with mergeSchema enabled, but pre-creating gives
# deterministic column ordering + NOT NULL on key fields)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {queries_table} (
    platform              STRING NOT NULL,
    scale                 STRING,
    warehouse_size        STRING,
    warehouse_size_type   STRING,
    max_clusters          INT,
    strategy_slug         STRING,
    optimization_strategy STRING,
    data_format           STRING,
    test_type             STRING,
    q_id                  STRING,        -- canonical query id, joinable across platforms
    label_raw             STRING,        -- original JMeter label (including stream suffix)
    stream_id             INT,           -- parsed stream number for concurrency runs
    start_time            TIMESTAMP,
    end_time              TIMESTAMP,
    duration_ms           LONG,
    success               BOOLEAN,
    response_code         STRING,
    response_message      STRING,
    thread_name           STRING,
    data_type             STRING,
    failure_message       STRING,
    bytes                 LONG,
    sent_bytes            LONG,
    url                   STRING,
    latency_ms            LONG,
    idle_time_ms          LONG,
    source_file           STRING,
    ingested_at           TIMESTAMP,
    benchmark_run_id      STRING
) USING DELTA
""")
print(f"Ensured {queries_table} exists")

In [ ]:
from pyspark.sql import functions as F


def _transform_batch(batch_df, batch_id):
    """Parse combo metadata from filename and normalise JMeter CSV columns."""
    if batch_df.isEmpty():
        return

    # Extract basename without "-report.csv"
    basename_expr = F.regexp_extract(F.col('_metadata.file_name'), r'^(.+?)-report\.csv$', 1)
    df = batch_df.withColumn("_basename", basename_expr)

    # Split on "-" : [prefix, platform, *test_type_parts]
    df = df.withColumn("_parts", F.split("_basename", "-"))
    df = df.withColumn("_prefix", F.col("_parts")[0])
    df = df.withColumn("platform", F.col("_parts")[1])
    df = df.withColumn("_test_type_raw",
        F.expr("concat_ws('-', slice(_parts, 3, size(_parts)))"))

    # Split prefix on "_" : [size, scale, *slug_parts]  (e.g. "M_100_NATIVE_GEN1")
    df = df.withColumn("_prefix_parts", F.split("_prefix", "_"))
    df = df.withColumn("warehouse_size", F.col("_prefix_parts")[0])
    df = df.withColumn("scale",          F.col("_prefix_parts")[1])
    # filename prefix is {SIZE}_{SCALE}_{MAXCLUSTERS}_{SLUG} (new) or {SIZE}_{SCALE}_{SLUG} (legacy)
    _mc_is_num = F.col("_prefix_parts")[2].rlike("^[0-9]+$")
    df = df.withColumn("max_clusters",
        F.when(_mc_is_num, F.col("_prefix_parts")[2].cast("int")).otherwise(F.lit(None).cast("int")))
    df = df.withColumn("strategy_slug",
        F.when(_mc_is_num, F.expr("concat_ws('_', slice(_prefix_parts, 4, size(_prefix_parts)))"))
         .otherwise(F.expr("concat_ws('_', slice(_prefix_parts, 3, size(_prefix_parts)))")))

    # _test_type_raw may carry a "__{run_id}" tag inserted by run_jmeter_with_tracking
    # (e.g. "ingestion__1234567890"). Split it off before normalising test_type.
    df = df.withColumn("benchmark_run_id",
        F.when(F.col("_test_type_raw").contains("__"),
               F.split(F.col("_test_type_raw"), "__")[1])
         .otherwise(F.lit(None).cast("string")))
    df = df.withColumn("_test_type_raw",
        F.split(F.col("_test_type_raw"), "__")[0])

    # Normalise test_type
    df = df.withColumn("test_type",
        F.when(F.col("_test_type_raw") == "perf-test", F.lit("power"))
         .otherwise(F.col("_test_type_raw")))

    # Derive data_format from strategy slug
    df = df.withColumn("data_format",
        F.when(F.col("platform") == "snowflake",
               F.when(F.upper(F.col("strategy_slug")).startswith("NATIVE_"),  F.lit("native"))
                .when(F.upper(F.col("strategy_slug")).startswith("ICEBERG_"), F.lit("iceberg"))
                .otherwise(F.lit("unknown")))
         .when(F.col("platform") == "databricks",
               F.when(F.upper(F.col("strategy_slug")).startswith("ICEBERG"), F.lit("iceberg"))
                .otherwise(F.lit("delta")))
         .otherwise(F.lit("unknown")))

    # Derive optimization_strategy (DBX's compaction/indexing concept; empty on SF)
    df = df.withColumn("optimization_strategy",
        F.when(F.col("platform") == "databricks",
               F.when(F.upper(F.col("strategy_slug")) == "LIQUID_CLUSTERING", F.lit("liquid_clustering"))
                .when(F.upper(F.col("strategy_slug")) == "ICEBERG_LIQUID_CLUSTERING",  F.lit("liquid_clustering"))
                .when(F.upper(F.col("strategy_slug")) == "ICEBERG",           F.lit("none"))
                .otherwise(F.lower(F.col("strategy_slug"))))
         .otherwise(F.lit("none")))

    # warehouse_size_type: serverless for DBX; gen1/gen2 for SF (inferred from slug suffix)
    df = df.withColumn("warehouse_size_type",
        F.when(F.col("platform") == "databricks", F.lit("serverless"))
         .when(F.upper(F.col("strategy_slug")).endswith("_ADAPTIVE"), F.lit("gen2_adaptive"))
         .when(F.upper(F.col("strategy_slug")).endswith("_GEN2"), F.lit("gen2"))
         .when(F.upper(F.col("strategy_slug")).endswith("_GEN1"), F.lit("gen1"))
         .otherwise(F.lit("unknown")))

    # Times + duration from JMeter's epoch-millis timeStamp + elapsed
    df = df.withColumn("duration_ms", F.col("elapsed").cast("long"))
    df = df.withColumn("start_time",
        (F.col("timeStamp").cast("long") / F.lit(1000)).cast("timestamp"))
    df = df.withColumn("end_time",
        ((F.col("timeStamp").cast("long") + F.col("elapsed").cast("long")) / F.lit(1000)).cast("timestamp"))

    # Rename JMeter columns to snake_case
    # Normalise JMeter label into canonical q_id (joinable across platforms) +
    # a separate stream_id for concurrency runs.
    df = df.withColumnRenamed("label", "label_raw")
    df = df.withColumn(
        "q_id",
        F.when(F.col("label_raw").rlike(r"^q\d+[a-z]?"),
               F.regexp_extract(F.col("label_raw"), r"^(q\d+[a-z]?)", 1))
         .when(F.col("label_raw").rlike(r"^(create|copy|optimize|analyze)_"),
               F.col("label_raw"))
         .otherwise(F.lit(None).cast("string"))
    )
    df = df.withColumn(
        "stream_id",
        F.when(F.col("label_raw").rlike(r"stream \d+"),
               F.regexp_extract(F.col("label_raw"), r"stream (\d+)", 1).cast("int"))
         .otherwise(F.lit(None).cast("int"))
    )
    df = df \
           .withColumnRenamed("responseCode", "response_code") \
           .withColumnRenamed("responseMessage", "response_message") \
           .withColumnRenamed("threadName", "thread_name") \
           .withColumnRenamed("dataType", "data_type") \
           .withColumnRenamed("failureMessage", "failure_message") \
           .withColumnRenamed("sentBytes", "sent_bytes") \
           .withColumnRenamed("URL", "url") \
           .withColumnRenamed("Latency", "latency_ms") \
           .withColumnRenamed("IdleTime", "idle_time_ms")

    # Tidy + cast types, then drop helper columns
    df = df.withColumn("source_file", F.col("_metadata.file_path"))
    df = df.withColumn("ingested_at", F.current_timestamp())

    # JMeter's CSV columns depend on saveservice defaults. Emit NULL for any
    # optional column missing from the current input schema.
    def _col(name, sql_type="string"):
        if name in df.columns:
            return F.col(name).cast(sql_type).alias(name)
        return F.lit(None).cast(sql_type).alias(name)

    final = df.select(
        F.col("platform"), F.col("scale"), F.col("warehouse_size"),
        F.col("warehouse_size_type"), F.col("max_clusters"), F.col("strategy_slug"),
        F.col("optimization_strategy"), F.col("data_format"),
        F.col("test_type"), F.col("q_id"), F.col("label_raw"), F.col("stream_id"),
        F.col("start_time"), F.col("end_time"), F.col("duration_ms"),
        _col("success", "boolean"),
        _col("response_code", "string"),
        _col("response_message", "string"),
        _col("thread_name", "string"),
        _col("data_type", "string"),
        _col("failure_message", "string"),
        _col("bytes", "long"),
        _col("sent_bytes", "long"),
        _col("url", "string"),
        _col("latency_ms", "long"),
        _col("idle_time_ms", "long"),
        F.col("source_file"), F.col("ingested_at"),
        F.col("benchmark_run_id"),
    )
    final.write.mode("append").option("mergeSchema", "true").saveAsTable(queries_table)
    _file_count = final.select("source_file").distinct().count()
    _row_count = final.count()
    print(f"batch {batch_id}: appended {_row_count} rows from {_file_count} file(s)")


# --- Autoloader read stream ---
source_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_dir)
        .option("cloudFiles.includeExistingFiles", "true")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("header", "true")
        .option("multiLine", "true")      # JMeter embeds stack traces with real newlines in failureMessage
        .option("escape", "\"")
        .load(results_path)
)

query = (
    source_stream.writeStream
        .option("checkpointLocation", checkpoint_dir)
        .foreachBatch(_transform_batch)
        .trigger(availableNow=True)
        .start()
)
query.awaitTermination()
print("Stage 1 complete")

In [ ]:
# Sanity — count rows per (platform, test_type) for visibility
display(
    spark.table(queries_table)
         .groupBy("platform", "test_type", "scale", "warehouse_size", "strategy_slug")
         .count()
         .orderBy("platform", "scale", "warehouse_size", "strategy_slug", "test_type")
)

### Stage 2 — Derive `results_prices`

One row per unique `(platform, warehouse_size, warehouse_size_type, region)`.
Strategy and data format do not affect price, so they're not part of the key.

The `MERGE` only inserts new configurations. Manual edits to `price_per_unit`
or `units_per_hour` persist across re-runs — the job never overwrites them.

Defaults: `price_per_unit = 0`, `units_per_hour = 0`. Units are DBUs for
Databricks, credits for Snowflake.


In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {prices_table} (
    platform            STRING NOT NULL,
    warehouse_size      STRING,
    warehouse_size_type STRING,
    region              STRING,
    tier                STRING,
    price_per_unit      DECIMAL(18, 6),
    units_per_hour      DECIMAL(18, 6),
    unit_label          STRING
) USING DELTA
""")
print(f"Ensured {prices_table} exists")

In [ ]:
from pyspark.sql import functions as F

runs = spark.table(queries_table)

configs = (
    runs.filter(F.col("platform").isNotNull()
                & F.col("warehouse_size").isNotNull()
                & F.col("warehouse_size_type").isNotNull())
        .select("platform", "warehouse_size", "warehouse_size_type")
        .distinct()
        .withColumn("region",         F.lit(region))
        .withColumn("tier",           F.lit(None).cast("string"))
        .withColumn("price_per_unit", F.lit(0).cast("decimal(18,6)"))
        .withColumn("units_per_hour", F.lit(0).cast("decimal(18,6)"))
        .withColumn("unit_label",
            F.when(F.col("platform") == "databricks", F.lit("DBU"))
             .when(F.col("platform") == "snowflake",  F.lit("credit"))
             .otherwise(F.lit("unit")))
)

configs.createOrReplaceTempView("_new_price_rows")

merge_stmt = f"""
MERGE INTO {prices_table} AS t
USING _new_price_rows AS s
   ON t.platform            <=> s.platform
  AND t.warehouse_size      <=> s.warehouse_size
  AND t.warehouse_size_type <=> s.warehouse_size_type
  AND t.region              <=> s.region
  AND t.tier                <=> s.tier
WHEN NOT MATCHED THEN
  INSERT (platform, warehouse_size, warehouse_size_type, region, tier,
          price_per_unit, units_per_hour, unit_label)
  VALUES (s.platform, s.warehouse_size, s.warehouse_size_type, s.region, s.tier,
          s.price_per_unit, s.units_per_hour, s.unit_label)
"""
spark.sql(merge_stmt)
print("Stage 2 complete")
display(spark.table(prices_table).orderBy("platform", "warehouse_size", "warehouse_size_type"))

### Notes

- **Idempotence.** Re-running this notebook only ingests files new since the
  last run (Autoloader checkpoint at `{volume}/autoloader/results/`). The
  `MERGE` on `results_prices` is insert-only against the compound key, so
  your edited prices are never overwritten.

- **Deleting a run.** Re-ingesting the same file is a no-op because Autoloader
  tracks file discovery. To force re-ingestion of a specific file, delete the
  file then drop the matching rows from `results_queries` before restoring the
  file. Or wipe the checkpoint to replay everything.

- **Filename contract.** Results files must land at
  `{volume}/results/{SIZE}_{SCALE}_{STRATEGY_SLUG}-{PLATFORM}-{TEST_TYPE}-report.csv`.
  The ingestion job silently skips files that don't match this pattern
  (they produce NULL metadata columns). Helper notebooks already write in
  this format.

- **`{{results_* tables skipped by cleanup}}`.** `7 - Databricks - Cleanup`
  ignores any table starting with `results_`, so rebuilding the benchmark
  schema across runs doesn't destroy your comparison history.
